# 03 — Baseline Models

Train single-task LightGBM models for each task to establish performance baselines.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.training.metrics import classification_metrics, regression_metrics
from src.utils.visualization import plot_confusion_matrix

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
train_df = pd.read_parquet('../data/processed/train.parquet')
val_df = pd.read_parquet('../data/processed/val.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')

label_encoder = joblib.load('../data/processed/label_encoder.joblib')

target_cols = ['class_label', 'h_target', 'diameter_target', 'h_mask', 'diameter_mask']
feature_cols = [c for c in train_df.columns if c not in target_cols]

X_train = train_df[feature_cols].values
X_val = val_df[feature_cols].values
X_test = test_df[feature_cols].values

print(f'Features: {len(feature_cols)}, Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

## 1. Classification Baseline — LightGBM

In [ ]:
y_cls_train = train_df['class_label'].values
y_cls_val = val_df['class_label'].values
y_cls_test = test_df['class_label'].values

clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
)

clf.fit(
    X_train, y_cls_train,
    eval_set=[(X_val, y_cls_val)],
    callbacks=[lgb.early_stopping(50, verbose=True)],
)

y_cls_pred = clf.predict(X_test)
print('\n=== Classification Results (Test Set) ===')
print(classification_report(y_cls_test, y_cls_pred, target_names=label_encoder.classes_))

cls_metrics = classification_metrics(y_cls_test, y_cls_pred)
print(f'Accuracy: {cls_metrics["accuracy"]:.4f}')
print(f'F1 Macro: {cls_metrics["f1_macro"]:.4f}')
print(f'F1 Weighted: {cls_metrics["f1_weighted"]:.4f}')

In [ ]:
fig = plot_confusion_matrix(y_cls_test, y_cls_pred, label_encoder.classes_.tolist())
plt.show()

## 2. H Magnitude Regression Baseline

In [ ]:
# Use only rows where H is present
h_train_mask = train_df['h_mask'].values.astype(bool)
h_test_mask = test_df['h_mask'].values.astype(bool)

reg_h = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    random_state=42,
    verbose=-1,
)

reg_h.fit(
    X_train[h_train_mask], train_df['h_target'].values[h_train_mask],
    eval_set=[(X_val[val_df['h_mask'].values.astype(bool)], 
               val_df['h_target'].values[val_df['h_mask'].values.astype(bool)])],
    callbacks=[lgb.early_stopping(50, verbose=True)],
)

y_h_pred = reg_h.predict(X_test[h_test_mask])
y_h_true = test_df['h_target'].values[h_test_mask]

h_metrics = regression_metrics(y_h_true, y_h_pred)
print('\n=== H Magnitude Regression Results (Test Set) ===')
for k, v in h_metrics.items():
    print(f'  {k}: {v:.4f}')

## 3. Diameter Regression Baseline

In [ ]:
# Use only rows where diameter is present
d_train_mask = train_df['diameter_mask'].values.astype(bool)
d_test_mask = test_df['diameter_mask'].values.astype(bool)

reg_d = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    random_state=42,
    verbose=-1,
)

reg_d.fit(
    X_train[d_train_mask], train_df['diameter_target'].values[d_train_mask],
    eval_set=[(X_val[val_df['diameter_mask'].values.astype(bool)],
               val_df['diameter_target'].values[val_df['diameter_mask'].values.astype(bool)])],
    callbacks=[lgb.early_stopping(50, verbose=True)],
)

y_d_pred = reg_d.predict(X_test[d_test_mask])
y_d_true = test_df['diameter_target'].values[d_test_mask]

d_metrics = regression_metrics(y_d_true, y_d_pred)
print('\n=== Diameter Regression Results (Test Set) ===')
for k, v in d_metrics.items():
    print(f'  {k}: {v:.4f}')

## 4. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, model, title in zip(axes, [clf, reg_h, reg_d], ['Classification', 'H Regression', 'Diameter Regression']):
    importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    importance.tail(15).plot(kind='barh', ax=ax)
    ax.set_title(f'{title} — Top 15 Features')

plt.tight_layout()
plt.show()

## 5. Baseline Summary

In [ ]:
print('=== Baseline Results Summary ===')
print(f'\nClassification:')
print(f'  Accuracy:    {cls_metrics["accuracy"]:.4f}')
print(f'  F1 (macro):  {cls_metrics["f1_macro"]:.4f}')
print(f'  F1 (weight): {cls_metrics["f1_weighted"]:.4f}')
print(f'\nH Magnitude Regression:')
for k, v in h_metrics.items():
    print(f'  {k:>4s}: {v:.4f}')
print(f'\nDiameter Regression (log-scale):')
for k, v in d_metrics.items():
    print(f'  {k:>4s}: {v:.4f}')